### 阶段一：数据清洗与目标变量计算

In [3]:
import pandas as pd
# 随便读取合并前的一个核心文件看一眼
df_temp1 = pd.read_csv('../data/merged_daily.csv', nrows=3)
df_temp2 = pd.read_csv('../data/merged_daily_gemini-1.5-flash_opinion.csv', nrows=3)
print(df_temp1.columns.tolist())
print(df_temp2.columns.tolist())

['timestamp', 'open', 'close', 'high', 'low', 'volume', 'blocks-size', 'avg-block-size', 'n-transactions-total', 'n-transactions-per-block', 'hash-rate', 'difficulty', 'miners-revenue', 'transaction-fees-usd', 'n-unique-addresses', 'n-transactions', 'estimated-transaction-volume-usd', 'total-bitcoins', 'market-cap', 'fng_value', 'fng_value_classification', 'fng_sentiment', 'cbbi_value', 'cbbi_sentiment', 'cointelegraph', 'bitcoin_news', 'reddit', 'avg_current_price', 'avg_next_price', 'pct_price_change', 'trend']
['timestamp', 'open', 'close', 'high', 'low', 'volume', 'blocks-size', 'avg-block-size', 'n-transactions-total', 'n-transactions-per-block', 'hash-rate', 'difficulty', 'miners-revenue', 'transaction-fees-usd', 'n-unique-addresses', 'n-transactions', 'estimated-transaction-volume-usd', 'total-bitcoins', 'market-cap', 'fng_value', 'fng_value_classification', 'fng_sentiment', 'cbbi_value', 'cbbi_sentiment', 'cointelegraph', 'bitcoin_news', 'reddit', 'avg_current_price', 'avg_next

In [ ]:
import pandas as pd
import numpy as np
import os


# ==========================================
# 1. 数据加载与合并
# ==========================================

# 确保读取路径符合本地项目文件结构规范
path_base = '../data/merged_daily.csv'
path_llm = '../data/merged_daily_gemini-1.5-flash_opinion.csv'

# 读取两份数据
df_base = pd.read_csv(path_base)
df_llm = pd.read_csv(path_llm)

# 将 timestamp 列转换为 datetime 格式
df_base['timestamp'] = pd.to_datetime(df_base['timestamp'])
df_llm['timestamp'] = pd.to_datetime(df_llm['timestamp'])

# 按时间正序排列（防止原始数据乱序导致时序特征错乱）
df_base = df_base.sort_values('timestamp').reset_index(drop=True)
df_llm = df_llm.sort_values('timestamp').reset_index(drop=True)

# 提取两份数据的公共列名。
# 因为 df_llm 包含了 df_base 的所有基础列，为避免合并时产生多余的 _x, _y 后缀，
# 我们直接对所有公共列和 timestamp 进行内连接合并（取交集），以保留完整的标注特征对齐数据。
common_cols = list(set(df_base.columns) & set(df_llm.columns))
df_merged = pd.merge(df_base, df_llm, on=common_cols, how='inner')

# ==========================================
# 2. 缺失值处理（严禁未来数据穿越）
# ==========================================

# 统一使用前向填充（ffill）来处理所有特征列的缺失值，利用历史最新可用数据填充
df_merged = df_merged.ffill()

# 如果最开头存在无法通过前向填充的 NaN（即第一天就有缺失），直接将其丢弃
df_merged = df_merged.dropna().reset_index(drop=True)

# ==========================================
# 3. 目标变量构建（核心）
# ==========================================

# 计算当天的 Parkinson 极值波动率 V_P
# 公式：V_P = (ln(H_t / L_t))^2 / (4 * ln(2))
# 注：在此步操作前，确保 high 和 low 不存在 <= 0 的异常值
ln_hl = np.log(df_merged['high'] / df_merged['low'])
v_p = (ln_hl ** 2) / (4 * np.log(2))

# 取自然对数，得到对数波动率 sigma_t
# 加入极小值 epsilon=1e-8，防止极端情况（某天高低点完全一样时 v_p=0）导致计算出 -inf，破坏后续网络训练
epsilon = 1e-8
sigma_t = np.log(v_p + epsilon)

# 防穿越后移：将当天的对数波动率向上平移一行，作为前一天的次日预测目标
df_merged['target_sigma_t_plus_1'] = sigma_t.shift(-1)

# 平移操作会导致最后一天缺少次日的数据（产生 NaN），必须删除这最后一行
df_merged = df_merged.dropna(subset=['target_sigma_t_plus_1']).reset_index(drop=True)

# ==========================================
# 4. 数据导出与结果校验
# ==========================================

# 检查目标文件夹是否存在，不存在则创建
os.makedirs('../data', exist_ok=True)

# 将预处理完的数据导出为 CSV，作为后续 PyTorch 建模的基准数据
output_path = '../data/01_preprocessed_data.csv'
df_merged.to_csv(output_path, index=False)

# 打印日志以供核验
print(f"数据预处理完成！")
print(f"文件已保存至: {output_path}")
print(f"合并并处理后的数据集 df.shape: {df_merged.shape}")
print("-" * 50)
print("最后几行数据展示（检查 target_sigma_t_plus_1 波动率是否成功错位向上平移）:")
print(df_merged[['timestamp', 'high', 'low', 'target_sigma_t_plus_1']].tail())

数据预处理完成！
文件已保存至: ../data/01_preprocessed_data.csv
合并并处理后的数据集 df.shape: (2337, 36)
--------------------------------------------------
最后几行数据展示（检查 target_sigma_t_plus_1 波动率是否成功错位向上平移）:
      timestamp     high      low  target_sigma_t_plus_1
2332 2024-06-24  63397.0  63052.0             -11.273785
2333 2024-06-25  60699.0  60340.0             -12.555604
2334 2024-06-26  61919.0  61726.0             -11.814030
2335 2024-06-27  61112.0  60836.0             -12.184682
2336 2024-06-28  61824.0  61592.0             -11.084055


### 阶段二：FinBERT 情感因子提取 (耗时警告！跑完后不再运行)

In [6]:
df_temp3 = pd.read_csv('../data/01_preprocessed_data.csv', nrows=3)
print(df_temp3.columns.tolist())

['timestamp', 'open', 'close', 'high', 'low', 'volume', 'blocks-size', 'avg-block-size', 'n-transactions-total', 'n-transactions-per-block', 'hash-rate', 'difficulty', 'miners-revenue', 'transaction-fees-usd', 'n-unique-addresses', 'n-transactions', 'estimated-transaction-volume-usd', 'total-bitcoins', 'market-cap', 'fng_value', 'fng_value_classification', 'fng_sentiment', 'cbbi_value', 'cbbi_sentiment', 'cointelegraph', 'bitcoin_news', 'reddit', 'avg_current_price', 'avg_next_price', 'pct_price_change', 'trend', 'reasoning_text', 'sentiment_class', 'action_class', 'action_score', 'target_sigma_t_plus_1']


In [7]:
import pandas as pd
import numpy as np
import torch
import ast
import gc
from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# 1. 硬件自适应配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"当前使用的设备: {device}")

# 2. 读取预处理后的数据
# 路径规范：从 ../data/ 读取
input_path = '../data/01_preprocessed_data.csv'
df = pd.read_csv(input_path)

# 3. 列表解析 (防坑处理)
# CSV 存储列表会变成字符串，使用 ast.literal_eval 还原为 Python List
list_columns = ['cointelegraph', 'bitcoin_news', 'reddit']
for col in list_columns:
    # 增加类型检查，防止重复运行报错或处理 NaN
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

print(f"数据加载完成，总行数: {df.shape[0]}")

当前使用的设备: cuda
数据加载完成，总行数: 2337


In [8]:
# 4. 加载 FinBERT 模型与分词器
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval() # 切换至推理模式

def get_sentiment_score(texts, batch_size=16):
    """
    核心情感计算逻辑 (方案 A)
    Score = P(Positive) - P(Negative)
    """
    if not texts:
        return 0.0
    
    scores = []
    # 按照 batch_size 进一步分段，严防大文本量导致 OOM
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        
        # 文本截断与对齐 (FinBERT 最大长度 512)
        inputs = tokenizer(batch_texts, padding=True, truncation=True, 
                           max_length=512, return_tensors="pt").to(device)
        
        with torch.no_grad(): # 禁用梯度计算，节省显存
            outputs = model(**inputs)
            # 对模型输出进行 Softmax 归一化得到概率
            probs = F.softmax(outputs.logits, dim=-1) 
            
            # FinBERT 标签映射通常为: 0: positive, 1: negative, 2: neutral
            # 注意：实际映射请以模型 config 为准，ProsusAI/finbert 默认为 [Pos, Neg, Neu]
            pos_probs = probs[:, 0].cpu().numpy()
            neg_probs = probs[:, 1].cpu().numpy()
            
            # 计算单条 Score = P(Pos) - P(Neg)
            batch_scores = pos_probs - neg_probs
            scores.extend(batch_scores.tolist())
            
        # 显存清理：手动删除 batch 变量并清理缓存
        del inputs, outputs, probs
        
    return np.mean(scores) if scores else 0.0

# 5. 执行每日文本聚合与情感打分
finbert_scores = []

# 使用 tqdm 打印外层日期进度条
for index, row in tqdm(df.iterrows(), total=len(df), desc="提取 FinBERT 情感因子"):
    # 聚合当天所有文本源
    daily_texts = []
    # reddit 结构较为特殊，通常是嵌套列表中的文本，这里假设已预处理为 List of strings
    # 如果 reddit 列是嵌套的 [author, text, score...]，需在此处取 index=1 的位置
    for col in list_columns:
        content = row[col]
        if isinstance(content, list):
            # 处理嵌套情况（如 reddit 常见的 [[user, text, ...], ...]）
            for item in content:
                if isinstance(item, list) and len(item) > 1:
                    daily_texts.append(str(item[4])) # 根据你提供的 sample50，正文在索引 4
                elif isinstance(item, str):
                    daily_texts.append(item)
    
    # 执行打分
    if not daily_texts:
        score = 0.0
    else:
        score = get_sentiment_score(daily_texts)
    
    finbert_scores.append(score)
    
    # 每一行处理完或每隔一定频率执行显存深度清理
    if index % 10 == 0:
        torch.cuda.empty_cache()
        gc.collect()

# 将结果填回 DataFrame
df['finbert_sentiment_score'] = finbert_scores

提取 FinBERT 情感因子: 100%|██████████| 2337/2337 [04:12<00:00,  9.25it/s]


In [9]:
# 6. 保存特征工程后的数据集
output_path = '../data/02_finbert_features_added.csv'
df.to_csv(output_path, index=False)

print("\n--- 特征提取完成 ---")
print(f"文件已保存至: {output_path}")

# 7. 检查前 10 行结果
print(df[['timestamp', 'finbert_sentiment_score']].head(10))

# 8. 技术上下文接力摘要
"""
【技术上下文接力摘要】
产出文件：../data/02_finbert_features_added.csv
Data Shape: (2337, 37)  # 新增了 finbert_sentiment_score
超参数：batch_size=16, max_length=512, model=ProsusAI/finbert
遗留问题：由于 reddit 原始数据中可能存在空列表或非字符串格式，已在代码中加入安全过滤。
         下一阶段将进行 LLM (Llama/Qwen) 情感因子提取。
"""


--- 特征提取完成 ---
文件已保存至: ../data/02_finbert_features_added.csv
    timestamp  finbert_sentiment_score
0  2018-02-01                -0.096697
1  2018-02-02                -0.079304
2  2018-02-03                -0.052735
3  2018-02-04                -0.068640
4  2018-02-05                -0.139091
5  2018-02-06                -0.082391
6  2018-02-07                -0.049396
7  2018-02-08                -0.105297
8  2018-02-09                -0.091154
9  2018-02-10                -0.062107


'\n【技术上下文接力摘要】\n产出文件：../data/02_finbert_features_added.csv\nData Shape: (2337, 37)  # 新增了 finbert_sentiment_score\n超参数：batch_size=16, max_length=512, model=ProsusAI/finbert\n遗留问题：由于 reddit 原始数据中可能存在空列表或非字符串格式，已在代码中加入安全过滤。\n         下一阶段将进行 LLM (Llama/Qwen) 情感因子提取。\n'

### 阶段三：Bi-LSTM 模型训练